# SNM Benchmark — Complete EDA

This notebook performs a data-first exploratory analysis of the three source tables, ground truth, train/test distribution, text noise, normalization behavior, duplicate/collision structure, and blocking requirements.

The EDA is designed for the existing SNM pipeline. It does not change the matching model or introduce country-specific rules.

Run this notebook from the project root containing `student_resource/` and `src/`.


In [ ]:
import os
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Working directory:", os.getcwd())
print("Top-level entries:", os.listdir(".")[:30])


## 1. Locate the dataset and load all tables

In [ ]:
candidates = [
    Path("student_resource/dataset"),
    Path("../student_resource/dataset"),
    Path("/home/oai/student_resource/dataset"),
]

BASE = next((p for p in candidates if p.exists()), None)

if BASE is None:
    raise FileNotFoundError(
        "Could not find student_resource/dataset. "
        "Run this notebook from the project root or update BASE manually."
    )

print("Dataset root:", BASE.resolve())

def read_tsv(path, **kwargs):
    return pd.read_csv(path, sep="\t", **kwargs)

s1 = read_tsv(BASE / "train/train_source1.tsv")
s2 = read_tsv(BASE / "train/train_source2.tsv")
s3 = read_tsv(BASE / "train/train_source3.tsv")
gt = read_tsv(BASE / "train/train_ground_truth.tsv", keep_default_na=False)

test_s1 = read_tsv(BASE / "test/test_source1.tsv")
test_s2 = read_tsv(BASE / "test/test_source2.tsv")
test_s3 = read_tsv(BASE / "test/test_source3.tsv")

tables = {
    "Train S1": s1,
    "Train S2": s2,
    "Train S3": s3,
    "Ground Truth": gt,
    "Test S1": test_s1,
    "Test S2": test_s2,
    "Test S3": test_s3,
}

for name, df in tables.items():
    print(f"{name:15s}: {df.shape}")


## 2. Schema, dtypes, and basic integrity

In [ ]:
for name, df in tables.items():
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print("Columns:", list(df.columns))
    print(df.dtypes)
    print("\nHead:")
    display(df.head(3))


In [ ]:
required_source_cols = {"entity_id", "business_name", "business_address", "country"}

for name, df in [("Train S1", s1), ("Train S2", s2), ("Train S3", s3),
                 ("Test S1", test_s1), ("Test S2", test_s2), ("Test S3", test_s3)]:
    missing = required_source_cols - set(df.columns)
    print(f"{name}: missing required columns -> {sorted(missing)}")

print("Ground-truth columns:", list(gt.columns))


## 3. Missing values and duplicate IDs

In [ ]:
def missing_report(df, name):
    out = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "dtype": df.dtypes.astype(str),
    })
    print(f"\n{name}")
    display(out.sort_values("missing_count", ascending=False))

for name, df in tables.items():
    missing_report(df, name)

print("\nDuplicate entity IDs")
for name, df in [("Train S1", s1), ("Train S2", s2), ("Train S3", s3),
                 ("Test S1", test_s1), ("Test S2", test_s2), ("Test S3", test_s3)]:
    dup = df["entity_id"].duplicated().sum()
    print(f"{name:10s}: {dup:,} duplicate IDs; unique={df['entity_id'].nunique():,}")


## 4. Exact duplicate records and repeated business fields

In [ ]:
for name, df in [("Train S1", s1), ("Train S2", s2), ("Train S3", s3),
                 ("Test S1", test_s1), ("Test S2", test_s2), ("Test S3", test_s3)]:
    non_id_cols = [c for c in df.columns if c != "entity_id"]
    exact_dup = df.duplicated(subset=non_id_cols).sum()
    name_dup = df["business_name"].duplicated(keep=False).sum()
    addr_dup = df["business_address"].duplicated(keep=False).sum()
    print(
        f"{name:10s}: exact(non-ID)={exact_dup:,}, "
        f"rows in repeated names={name_dup:,}, "
        f"rows in repeated addresses={addr_dup:,}"
    )


## 5. Country distribution and train/test distribution shift

In [ ]:
def country_summary(df, label):
    x = df["country"].astype("string").fillna("<NA>").value_counts(dropna=False)
    out = pd.DataFrame({"count": x, "pct": x / len(df) * 100})
    out.index.name = "country"
    out["dataset"] = label
    return out.reset_index()

country_all = pd.concat([
    country_summary(s1, "Train S1"),
    country_summary(s2, "Train S2"),
    country_summary(s3, "Train S3"),
    country_summary(test_s1, "Test S1"),
    country_summary(test_s2, "Test S2"),
    country_summary(test_s3, "Test S3"),
], ignore_index=True)

display(country_all.sort_values(["dataset", "count"], ascending=[True, False]).head(100))

for label, df in [("Train S1", s1), ("Test S1", test_s1)]:
    print(f"\n{label} top countries:")
    display(country_summary(df, label).head(20))


In [ ]:
train_country = s1["country"].value_counts(normalize=True).mul(100)
test_country = test_s1["country"].value_counts(normalize=True).mul(100)

country_compare = pd.concat(
    [train_country.rename("train_pct"), test_country.rename("test_pct")],
    axis=1
).fillna(0).sort_values("test_pct", ascending=False)

display(country_compare.head(30))

country_compare.head(20).plot(kind="bar", figsize=(14, 5))
plt.ylabel("Share (%)")
plt.title("S1 Train vs Test Country Distribution")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 6. Ground-truth structure

In [ ]:
gt = gt.copy()
gt["matched_ids"] = gt["matched_entity_ids"].apply(
    lambda x: [v.strip() for v in str(x).split(",") if v.strip()]
)
gt["match_count"] = gt["matched_ids"].str.len()

print("Ground-truth rows:", len(gt))
print("Unique S1 IDs in ground truth:", gt["source1_entity_id"].nunique())
print("Rows with zero matches:", (gt["match_count"] == 0).sum())
print("Rows with >=1 match:", (gt["match_count"] > 0).sum())
print("No-match rate: %.2f%%" % ((gt["match_count"] == 0).mean() * 100))
print("Mean matches per S1 entity: %.4f" % gt["match_count"].mean())

display(gt["match_count"].value_counts().sort_index().rename("entities").to_frame())

plt.figure(figsize=(10, 4))
gt["match_count"].clip(upper=20).value_counts().sort_index().plot(kind="bar")
plt.xlabel("Number of matched records (20 = 20+)")
plt.ylabel("S1 entities")
plt.title("Ground-truth Match Count Distribution")
plt.tight_layout()
plt.show()


## 7. S2/S3 coverage and ground-truth integrity checks

In [ ]:
def source_counts(ids):
    s2c = sum(x.startswith("S2-") for x in ids)
    s3c = sum(x.startswith("S3-") for x in ids)
    other = [x for x in ids if not (x.startswith("S2-") or x.startswith("S3-"))]
    return s2c, s3c, other

counts = gt["matched_ids"].map(source_counts)
gt["s2_count"] = counts.map(lambda x: x[0])
gt["s3_count"] = counts.map(lambda x: x[1])
gt["other_match_count"] = counts.map(lambda x: len(x[2]))

summary = {
    "S1 entities": len(gt),
    "with S2 match": int((gt["s2_count"] > 0).sum()),
    "with S3 match": int((gt["s3_count"] > 0).sum()),
    "with both S2 and S3": int(((gt["s2_count"] > 0) & (gt["s3_count"] > 0)).sum()),
    "S2 match links": int(gt["s2_count"].sum()),
    "S3 match links": int(gt["s3_count"].sum()),
    "unexpected match ID prefixes": int((gt["other_match_count"] > 0).sum()),
}
display(pd.Series(summary, name="value").to_frame())

s2_ids = set(s2["entity_id"])
s3_ids = set(s3["entity_id"])
gt_s1_ids = set(gt["source1_entity_id"])
all_gt_ids = set(x for ids in gt["matched_ids"] for x in ids)

print("GT S1 IDs missing from train S1:", len(gt_s1_ids - set(s1["entity_id"])))
print("GT S2 IDs missing from train S2:", len({x for x in all_gt_ids if x.startswith('S2-')} - s2_ids))
print("GT S3 IDs missing from train S3:", len({x for x in all_gt_ids if x.startswith('S3-')} - s3_ids))
print("Duplicate source1_entity_id rows in GT:", gt["source1_entity_id"].duplicated().sum())

matched_id_series = pd.Series([x for ids in gt["matched_ids"] for x in ids])
print("Matched entity IDs repeated across GT rows:", matched_id_series.duplicated().sum())


## 8. Actual matched-pair inspection

In [ ]:
s1_lookup = s1.set_index("entity_id")
s2_lookup = s2.set_index("entity_id")
s3_lookup = s3.set_index("entity_id")

matched_gt = gt[gt["match_count"] > 0]
sample_n = min(20, len(matched_gt))
sample = matched_gt.sample(sample_n, random_state=42)

for row in sample.itertuples():
    sid = row.source1_entity_id
    s1_rec = s1_lookup.loc[sid]
    print(f"\n--- {sid} ---")
    print(f"S1 name : {s1_rec['business_name']!r}")
    print(f"S1 addr : {s1_rec['business_address']!r}")
    print(f"S1 country: {s1_rec['country']!r}")
    for mid in row.matched_ids:
        rec = s2_lookup.loc[mid] if mid.startswith("S2-") else s3_lookup.loc[mid]
        print(f"  {mid} name: {rec['business_name']!r}")
        print(f"  {mid} addr: {rec['business_address']!r}")
        print(f"  {mid} country: {rec['country']!r}")


## 9. Text length, token count, and character-level noise

In [ ]:
def text_profile(df, label):
    out = pd.DataFrame(index=df.index)
    for col in ["business_name", "business_address"]:
        s = df[col].fillna("").astype(str)
        out[f"{col}_chars"] = s.str.len()
        out[f"{col}_tokens"] = s.str.split().str.len()
        out[f"{col}_digits"] = s.str.count(r"\d")
        out[f"{col}_nonascii"] = s.map(lambda x: sum(ord(ch) > 127 for ch in x))
    summary = out.agg(["count", "mean", "median", "std", "min", "max"]).T
    summary["dataset"] = label
    return summary.reset_index(names="feature")

profiles = pd.concat([
    text_profile(s1, "Train S1"),
    text_profile(s2, "Train S2"),
    text_profile(s3, "Train S3"),
    text_profile(test_s1, "Test S1"),
    text_profile(test_s2, "Test S2"),
    text_profile(test_s3, "Test S3"),
], ignore_index=True)

display(profiles)


In [ ]:
def pct_non_ascii(series):
    s = series.fillna("").astype(str)
    return s.map(lambda x: any(ord(ch) > 127 for ch in x)).mean() * 100

for name, df in [("Train S1", s1), ("Train S2", s2), ("Train S3", s3),
                 ("Test S1", test_s1), ("Test S2", test_s2), ("Test S3", test_s3)]:
    print(
        f"{name:10s}: "
        f"name non-ASCII={pct_non_ascii(df['business_name']):.2f}%, "
        f"address non-ASCII={pct_non_ascii(df['business_address']):.2f}%"
    )

fr = test_s1[test_s1["country"].astype(str).str.casefold().eq("france")]
print("\nFrance rows in Test S1:", len(fr))
if len(fr):
    print("Sample France rows:")
    display(fr[["business_name", "business_address", "country"]].sample(min(10, len(fr)), random_state=1))


## 10. Common address-noise patterns

In [ ]:
def pattern_report(df, label):
    s = df["business_address"].fillna("").astype(str)
    patterns = {
        "near/landmark word": s.str.contains(r"\bnear\b|\bopposite\b|\bbehind\b|\bnext to\b", case=False, regex=True),
        "5-6 digit number": s.str.contains(r"\b\d{5,6}\b", regex=True),
        "any digit": s.str.contains(r"\d", regex=True),
        "punctuation": s.str.contains(r"[^A-Za-z0-9\s]", regex=True),
        "multiple spaces": s.str.contains(r"\s{2,}", regex=True),
        "slash": s.str.contains("/", regex=False),
        "hyphen": s.str.contains("-", regex=False),
        "comma": s.str.contains(",", regex=False),
    }
    return pd.Series({k: v.mean() * 100 for k, v in patterns.items()}, name=label)

address_patterns = pd.concat([
    pattern_report(s1, "Train S1"),
    pattern_report(s2, "Train S2"),
    pattern_report(s3, "Train S3"),
    pattern_report(test_s1, "Test S1"),
    pattern_report(test_s2, "Test S2"),
    pattern_report(test_s3, "Test S3"),
], axis=1)

display(address_patterns.round(2))

print("\nSample noisy S2 addresses:")
display(s2.loc[
    s2["business_address"].fillna("").str.contains(
        r"\bnear\b|\bopposite\b|\bbehind\b|\bnext to\b", case=False, regex=True
    ),
    ["business_name", "business_address", "country"]
].head(20))


## 11. Normalization behavior used by the existing pipeline

In [ ]:
sys.path.insert(0, ".")
import importlib
import src.normalize as normalize_module
normalize_module = importlib.reload(normalize_module)

from src.normalize import normalize_name, normalize_address, name_tokens, address_tokens, sorted_token_key

print("sorted_token_key test:", sorted_token_key({"techinfra", "gangaya", "limited"}))
print("normalize_name test:", normalize_name("Gangaya Techinfra Limited"))
print("normalize_address test:", normalize_address("Flat No.-053, Main Road"))


In [ ]:
def normalization_report(df):
    x = df.copy()
    x["norm_name"] = x["business_name"].map(normalize_name)
    x["norm_address"] = x["business_address"].map(normalize_address)
    x["name_key"] = x["norm_name"].map(lambda v: sorted_token_key(set(v.split())) if v else "")
    x["address_key"] = x["norm_address"].map(lambda v: sorted_token_key(set(v.split())) if v else "")
    return x

s1_norm = normalization_report(s1)
s2_norm = normalization_report(s2)
s3_norm = normalization_report(s3)
test_s1_norm = normalization_report(test_s1)
test_s2_norm = normalization_report(test_s2)
test_s3_norm = normalization_report(test_s3)

for label, df in [("Train S1", s1_norm), ("Train S2", s2_norm), ("Train S3", s3_norm),
                  ("Test S1", test_s1_norm), ("Test S2", test_s2_norm), ("Test S3", test_s3_norm)]:
    print(f"\n{label}")
    print("Rows:", len(df))
    print("Empty normalized names:", int((df["norm_name"] == "").sum()))
    print("Empty normalized addresses:", int((df["norm_address"] == "").sum()))
    print("Repeated normalized names:", int((df["norm_name"].value_counts() > 1).sum()))
    print("Repeated country+normalized-name keys:",
          int((df["country"].astype(str) + "|" + df["norm_name"]).value_counts().gt(1).sum()))
    print("Repeated country+name+address keys:",
          int((df["country"].astype(str) + "|" + df["norm_name"] + "|" + df["norm_address"]).value_counts().gt(1).sum()))


## 12. Collision analysis — ambiguity of normalized keys

In [ ]:
def collision_table(df, key_cols):
    key = df[key_cols].astype(str).agg("|".join, axis=1)
    counts = key.value_counts()
    return pd.Series({
        "rows": len(df),
        "unique_keys": counts.size,
        "keys_with_2+": int((counts >= 2).sum()),
        "rows_in_2+ groups": int(counts[counts >= 2].sum()),
        "largest_group": int(counts.max()) if len(counts) else 0,
        "mean_group_size_among_collisions": float(counts[counts >= 2].mean()) if (counts >= 2).any() else 0.0,
    })

rows = []
for label, df in [("S1 train", s1_norm), ("S2 train", s2_norm), ("S3 train", s3_norm),
                  ("S1 test", test_s1_norm), ("S2 test", test_s2_norm), ("S3 test", test_s3_norm)]:
    for key_name, cols in [
        ("country+name", ["country", "norm_name"]),
        ("country+sorted_name", ["country", "name_key"]),
        ("country+name+address", ["country", "norm_name", "norm_address"]),
        ("country+sorted_name+sorted_address", ["country", "name_key", "address_key"]),
    ]:
        r = collision_table(df, cols)
        r["dataset"] = label
        r["key"] = key_name
        rows.append(r)

collision_df = pd.DataFrame(rows)
display(collision_df[[
    "dataset", "key", "keys_with_2+", "rows_in_2+ groups",
    "largest_group", "mean_group_size_among_collisions"
]])


## 13. Ground-truth matched-pair normalization diagnostics

This section samples true links rather than performing an all-pairs comparison, keeping the EDA practical at millions of records.


In [ ]:
pair_rows = []

for row in matched_gt.itertuples():
    for mid in row.matched_ids:
        pair_rows.append((row.source1_entity_id, mid))
        if len(pair_rows) >= 20000:
            break
    if len(pair_rows) >= 20000:
        break

pairs = pd.DataFrame(pair_rows, columns=["s1_id", "match_id"])

def fetch_record(mid):
    if mid.startswith("S2-"):
        return s2_lookup.loc[mid]
    return s3_lookup.loc[mid]

if len(pairs):
    left = s1_lookup.loc[pairs["s1_id"], ["business_name", "business_address", "country"]].reset_index()
    left.columns = ["s1_id", "s1_name", "s1_address", "s1_country"]

    right_rows = []
    for mid in pairs["match_id"]:
        rec = fetch_record(mid)
        right_rows.append([mid, rec["business_name"], rec["business_address"], rec["country"]])
    right = pd.DataFrame(
        right_rows,
        columns=["match_id", "match_name", "match_address", "match_country"]
    )

    pair_diag = pairs.merge(left, on="s1_id").merge(right, on="match_id")
    pair_diag["s1_norm_name"] = pair_diag["s1_name"].map(normalize_name)
    pair_diag["match_norm_name"] = pair_diag["match_name"].map(normalize_name)
    pair_diag["s1_norm_addr"] = pair_diag["s1_address"].map(normalize_address)
    pair_diag["match_norm_addr"] = pair_diag["match_address"].map(normalize_address)

    pair_diag["name_exact_raw"] = (
        pair_diag["s1_name"].fillna("").astype(str).str.casefold()
        == pair_diag["match_name"].fillna("").astype(str).str.casefold()
    )
    pair_diag["name_exact_normalized"] = pair_diag["s1_norm_name"].eq(pair_diag["match_norm_name"])
    pair_diag["address_exact_normalized"] = pair_diag["s1_norm_addr"].eq(pair_diag["match_norm_addr"])
    pair_diag["country_same"] = pair_diag["s1_country"].eq(pair_diag["match_country"])

    print("Sampled true links:", len(pair_diag))
    display(
        pair_diag[
            ["name_exact_raw", "name_exact_normalized",
             "address_exact_normalized", "country_same"]
        ].mean().mul(100).round(2).rename("percentage").to_frame()
    )
else:
    pair_diag = pd.DataFrame()
    print("No matched links available.")


## 14. Token-reordering evidence

The existing pipeline contains `sorted_token_key()` specifically to make blocking invariant to word order. This checks how often true links have the same normalized token set while their ordered normalized strings differ.


In [ ]:
if len(pair_diag):
    def token_set(v):
        return set(str(v).split()) if str(v).strip() else set()

    pair_diag["s1_tokens"] = pair_diag["s1_norm_name"].map(token_set)
    pair_diag["match_tokens"] = pair_diag["match_norm_name"].map(token_set)
    pair_diag["same_token_set"] = pair_diag["s1_tokens"].eq(pair_diag["match_tokens"])
    pair_diag["same_ordered_norm"] = pair_diag["s1_norm_name"].eq(pair_diag["match_norm_name"])

    print("True links with identical normalized token set:",
          f"{pair_diag['same_token_set'].mean()*100:.2f}%")
    print("True links with different normalized order:",
          f"{(pair_diag['same_token_set'] & ~pair_diag['same_ordered_norm']).mean()*100:.2f}%")

    display(pair_diag[
        pair_diag["same_token_set"] & ~pair_diag["same_ordered_norm"]
    ][["s1_name", "match_name", "s1_norm_name", "match_norm_name"]].head(20))


## 15. Non-ASCII and transliteration risk by country

In [ ]:
def country_nonascii(df):
    x = df.copy()
    x["name_nonascii"] = x["business_name"].fillna("").astype(str).map(
        lambda v: any(ord(c) > 127 for c in v)
    )
    x["addr_nonascii"] = x["business_address"].fillna("").astype(str).map(
        lambda v: any(ord(c) > 127 for c in v)
    )
    return x.groupby("country").agg(
        rows=("entity_id", "size"),
        name_nonascii_pct=("name_nonascii", "mean"),
        address_nonascii_pct=("addr_nonascii", "mean"),
    ).assign(
        name_nonascii_pct=lambda d: d["name_nonascii_pct"] * 100,
        address_nonascii_pct=lambda d: d["address_nonascii_pct"] * 100,
    ).sort_values("rows", ascending=False)

display(country_nonascii(test_s1).head(50))


## 16. Entity-ID prefix and source-overlap checks

In [ ]:
for label, df in [("S1 train", s1), ("S2 train", s2), ("S3 train", s3),
                  ("S1 test", test_s1), ("S2 test", test_s2), ("S3 test", test_s3)]:
    prefixes = df["entity_id"].astype(str).str.extract(r"^([^-]+-)")[0].fillna("<none>").value_counts()
    print(f"\n{label} ID prefixes:")
    print(prefixes.to_dict())

sets = {name: set(df["entity_id"]) for name, df in [
    ("S1", s1), ("S2", s2), ("S3", s3)
]}
print("\nCross-source ID overlaps:")
for a, b in [("S1", "S2"), ("S1", "S3"), ("S2", "S3")]:
    print(f"{a} ∩ {b}:", len(sets[a] & sets[b]))


## 17. Candidate-blocking quality on a manageable sample

This is a pipeline diagnostic, not a replacement for the final benchmark. It measures recall and candidate-set size using the existing `src.blocking` implementation.


In [ ]:
RUN_BLOCKING_DIAGNOSTIC = True
BLOCKING_SAMPLE_SIZE = 2000
BLOCKING_COUNTRY = "US"

if RUN_BLOCKING_DIAGNOSTIC:
    from src.blocking import generate_candidates, blocking_quality_report

    s1_sample = s1[s1["country"].astype(str).eq(BLOCKING_COUNTRY)]
    s1_sample = s1_sample.sample(min(BLOCKING_SAMPLE_SIZE, len(s1_sample)), random_state=42)

    gt_dict = {row.source1_entity_id: row.matched_ids for row in gt.itertuples()}

    start = time.time()
    candidates = generate_candidates(
        s1_sample, s2, s3,
        window=15,
        max_candidates=20,
    )
    elapsed = time.time() - start

    gt_sample = {k: gt_dict[k] for k in s1_sample["entity_id"] if k in gt_dict}
    report = blocking_quality_report(
        candidates,
        gt_sample,
        n_s2_s3=len(s2) + len(s3),
    )

    print(f"Sample size: {len(s1_sample):,}")
    print(f"Blocking time: {elapsed:.1f}s")
    display(pd.Series(report).to_frame("value"))
else:
    print("Blocking diagnostic skipped.")


## 18. Final EDA summary

In [ ]:
print("DATASET SIZE")
for name, df in tables.items():
    print(f"  {name:15s}: {len(df):,} rows")

print("\nGROUND TRUTH")
print(f"  S1 entities in GT : {len(gt):,}")
print(f"  no-match entities : {(gt['match_count'] == 0).sum():,}")
print(f"  matched entities  : {(gt['match_count'] > 0).sum():,}")
print(f"  mean matches/S1   : {gt['match_count'].mean():.4f}")

print("\nTEXT / NORMALIZATION")
for label, df in [("S1 train", s1_norm), ("S2 train", s2_norm), ("S3 train", s3_norm),
                  ("S1 test", test_s1_norm)]:
    repeated = (df["country"].astype(str) + "|" + df["norm_name"]).value_counts().gt(1).sum()
    print(
        f"  {label:10s}: empty norm names={int((df['norm_name']=='').sum()):,}, "
        f"repeated country+name={int(repeated):,}"
    )

print("\nINTERPRETATION FOR PIPELINE DESIGN")
print("  1. Exact string matching is insufficient when true links contain formatting/noise differences.")
print("  2. Normalization should remain generic and country-independent.")
print("  3. Country + normalized name is useful but can collide, so address/token features are needed.")
print("  4. Sorted-token keys address business-name word-order changes and are useful for blocking.")
print("  5. Address landmarks, punctuation, missing values, numeric formatting, and non-ASCII text")
print("     must be handled without assuming one country's format.")
print("  6. Candidate generation should be evaluated by recall as well as candidate-set size.")
